# Basics

The shortest path to a probability, and the conventions everything else in these notebooks assumes.

**NuOscProbExact** computes oscillation probabilities *exactly*, with no approximation beyond floating-point round-off, for any time-independent Hamiltonian. The method expands the Hamiltonian and the evolution operator in the SU(2), SU(3) and SU(4) bases, which gives closed forms rather than a numerical integration.

In [1]:
import sys
import os

# Works whether or not the package is installed: these notebooks sit one
# level below the repository root, so src/ is one directory up.
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

import numpy as np
import matplotlib.pyplot as plt

import globaldefs as gd
import oscprob2nu
import oscprob3nu
import oscprob4nu
import hamiltonians2nu
import hamiltonians3nu
import hamiltonians4nu

%matplotlib inline
plt.rcParams.update({'figure.figsize': (7.2, 4.2), 'figure.dpi': 90,
                     'axes.grid': True, 'grid.alpha': 0.3,
                     'font.size': 11, 'legend.frameon': False,
                     # Axes end exactly at the data: no padding beyond
                     # x_min/x_max or y_min/y_max.
                     'axes.xmargin': 0.0, 'axes.ymargin': 0.0})

# The three-flavor vacuum Hamiltonian at the NuFit best fit, normal ordering.
# It is energy-independent; dividing by the energy is what the builders do.
H_VAC_3NU = hamiltonians3nu.hamiltonian_3nu_vacuum_energy_independent(
    gd.S12_NO_BF, gd.S23_NO_BF, gd.S13_NO_BF, gd.DCP_NO_BF,
    gd.D21_NO_BF, gd.D31_NO_BF)

KM = gd.CONV_KM_TO_INV_EV      # multiply a length in km to get eV^-1
GEV = 1.0e9                    # multiply an energy in GeV to get eV

## Units

The library is unit-agnostic: it asks only that the Hamiltonian and the baseline be given in reciprocal units, so that $HL$ is dimensionless. Everything below uses **eV** for energies and **eV$^{-1}$** for baselines, with `globaldefs` supplying the conversions.

In [2]:
print("1 km          = %.4e eV^-1" % KM)
print("1 GeV         = %.0e eV" % GEV)
print("crust V_CC    = %.4e eV" % gd.VCC_EARTH_CRUST)

1 km          = 5.0677e+09 eV^-1
1 GeV         = 1e+09 eV
crust V_CC    = 1.1358e-13 eV


## What the Hamiltonian is

Every routine here takes a Hamiltonian — a Hermitian matrix — and a baseline. If you have not met the oscillation Hamiltonian before, this is the one piece of physics worth having up front, because everything else in these notebooks is a variation on it.

Neutrinos are produced and detected as **flavor** states ($\nu_e, \nu_\mu, \nu_\tau$) but propagate as **mass** states ($\nu_1, \nu_2, \nu_3$). The two bases are related by the PMNS matrix $U$ — three angles $\theta_{12}, \theta_{13}, \theta_{23}$ and a CP-violating phase $\delta_{CP}$ — and the Hamiltonian is diagonal in the mass basis, so in the flavor basis it reads

$$ H_{\rm vac} = \frac{1}{2E}\, U M^2 U^\dagger , \qquad M^2 = \mathrm{diag}\left(0,\, \Delta m^2_{21},\, \Delta m^2_{31}\right) . $$

Only mass-squared *differences* appear, which is why the first entry can be set to zero. Matter adds a potential that the electron flavor feels and the others do not,

$$ H = H_{\rm vac} + \mathrm{diag}(V_{CC},\, 0,\, 0) , $$

and that is the whole content. Non-standard interactions, Lorentz-invariance violation and a sterile state are each a different Hermitian matrix in that same slot, evaluated by the same routine.

`H_VAC_3NU` in the setup cell is exactly $U M^2 U^\dagger/2$ at the NuFit 4.0 best fit for the normal ordering. Building it by hand shows there is nothing hidden in the builder:

In [3]:
# The mixing angles, as sin^2(theta) -- which is how they are
# quoted -- and the phase in degrees.
print("sin^2(th12) = %.3f" % gd.S12_NO_BF**2)
print("sin^2(th23) = %.3f" % gd.S23_NO_BF**2)
print("sin^2(th13) = %.5f" % gd.S13_NO_BF**2)
print("delta_CP    = %.0f degrees" % np.degrees(gd.DCP_NO_BF))
print("dm21^2      = %.3e eV^2" % gd.D21_NO_BF)
print("dm31^2      = %.3e eV^2" % gd.D31_NO_BF)

# Note: the builders take sin(theta), not theta itself.
U = np.asarray(hamiltonians3nu.pmns_mixing_matrix(
    gd.S12_NO_BF, gd.S23_NO_BF, gd.S13_NO_BF, gd.DCP_NO_BF))
M2 = np.diag([0.0, gd.D21_NO_BF, gd.D31_NO_BF]).astype(complex)

by_hand = U @ M2 @ U.conj().T / 2.0
reference = np.asarray(H_VAC_3NU)
print("\nU M^2 U^dag / 2  vs  H_VAC_3NU")
print("  absolute : %.2e eV^2"
      % np.max(np.abs(by_hand - reference)))
print("  relative : %.2e"
      % (np.max(np.abs(by_hand - reference))
         / np.max(np.abs(reference))))

sin^2(th12) = 0.310
sin^2(th23) = 0.582
sin^2(th13) = 0.02240
delta_CP    = 217 degrees
dm21^2      = 7.390e-05 eV^2
dm31^2      = 2.525e-03 eV^2

U M^2 U^dag / 2  vs  H_VAC_3NU
  absolute : 1.08e-19 eV^2
  relative : 1.48e-16


## One three-flavor probability

A 1 GeV neutrino travelling 1300 km in vacuum — roughly the DUNE baseline. The nine probabilities come back with the *initial* flavor varying slowest.

In [4]:
energy = 1.0*GEV
baseline = 1300.0*KM

# The vacuum Hamiltonian at this energy
H = np.asarray(H_VAC_3NU)/energy

prob = oscprob3nu.probabilities_3nu(H, baseline)

flavors = ["e", "mu", "tau"]
for k, value in enumerate(prob):
    a, b = divmod(k, 3)
    print("P_%-7s = %.6f"
          % (flavors[a]+flavors[b], value))

P_ee      = 0.927678
P_emu     = 0.014323
P_etau    = 0.057999
P_mue     = 0.040227
P_mumu    = 0.378872
P_mutau   = 0.580901
P_taue    = 0.032095
P_taumu   = 0.606804
P_tautau  = 0.361100


### Reading the nine numbers

The convention, used everywhere in this library and in every notebook after this one:

$$ P_{\alpha\beta} \equiv P(\nu_\alpha \to \nu_\beta) = \left|[U(L)]_{\beta\alpha}\right|^2 $$

so the **initial** flavor varies slowest. Later notebooks index the result directly — `p[:, 3]` for $P_{\mu e}$, `p[:, 4]` for $P_{\mu\mu}$ — and this is the map they are using:

In [5]:
print("index   from      to        symbol")
for k in range(9):
    a, b = divmod(k, 3)
    print("  %d     nu_%-4s -> nu_%-5s  P_%s%s"
          % (k, flavors[a], flavors[b],
             flavors[a], flavors[b]))

index   from      to        symbol
  0     nu_e    -> nu_e      P_ee
  1     nu_e    -> nu_mu     P_emu
  2     nu_e    -> nu_tau    P_etau
  3     nu_mu   -> nu_e      P_mue
  4     nu_mu   -> nu_mu     P_mumu
  5     nu_mu   -> nu_tau    P_mutau
  6     nu_tau  -> nu_e      P_taue
  7     nu_tau  -> nu_mu     P_taumu
  8     nu_tau  -> nu_tau    P_tautau


Each initial flavor conserves probability, which is the first thing to check of any oscillation code:

In [6]:
prob = np.array(prob)
for start, flavor in zip((0, 3, 6), ("e", "mu", "tau")):
    print("sum over final flavors, from nu_%-4s = %.15f"
          % (flavor, prob[start:start+3].sum()))

sum over final flavors, from nu_e    = 1.000000000000002
sum over final flavors, from nu_mu   = 1.000000000000000
sum over final flavors, from nu_tau  = 1.000000000000000


## The trace does not matter

Adding a multiple of the identity to $H$ shifts every eigenvalue by the same amount, so it contributes an overall phase that cancels in $|U_{\beta\alpha}|^2$. The library drops the trace internally and works with the traceless part throughout.

This is worth knowing because it means an overall energy offset in your Hamiltonian — a flavor-universal potential, for instance — can simply be left out.

In [7]:
shift = 3.0*np.max(np.abs(H))     # comparable to H itself

p_plain = np.array(oscprob3nu.probabilities_3nu(H, baseline))
p_shift = np.array(oscprob3nu.probabilities_3nu(
    H + shift*np.eye(3), baseline))

print("max |P(H) - P(H + c*1)| = %.2e"
      % np.max(np.abs(p_plain - p_shift)))

max |P(H) - P(H + c*1)| = 1.57e-14


One caveat, since someone will try it: that holds *arithmetically* for any shift, but not numerically for an absurd one. The traceless part is recovered by subtracting numbers of order the shift, so adding $10^{10}\,|H|$ leaves only a few significant digits of the physics and the agreement degrades to about $10^{-7}$. That is ordinary floating-point cancellation rather than anything about the method — but if your Hamiltonian carries a huge flavor-universal term, subtract it yourself before passing it in.

## Two flavors

The same interface, with four probabilities instead of nine.

In [8]:
# The builders take sin(theta), not theta itself.
s12 = np.sqrt(0.310)
H2_vac = hamiltonians2nu.hamiltonian_2nu_vacuum_energy_independent(
    s12, gd.D21_NO_BF)

Pee, Pem, Pme, Pmm = oscprob2nu.probabilities_2nu(
    np.asarray(H2_vac)/energy, baseline)
print("Pee = %.6f   Pem = %.6f" % (Pee, Pem))

Pee = 0.987387   Pem = 0.012613


## Any Hermitian matrix

Nothing above is special. The core routines take an arbitrary Hermitian matrix, which is what makes the library useful for non-standard scenarios: new interactions, Lorentz-invariance violation and sterile-like perturbations are all just entries in that matrix.

The matrix below is not any physical scenario — it is a reminder that the routine neither knows nor needs to know where the numbers came from.

In [9]:
H_arbitrary = np.array([[0.0, 1.0+2.0j, 0.5],
                        [1.0-2.0j, 1.0, 0.0],
                        [0.5, 0.0, -1.0]], dtype=complex)

p_arb = oscprob3nu.probabilities_3nu(H_arbitrary, 1.0)

for k, value in enumerate(p_arb):
    a, b = divmod(k, 3)
    print("P_%s%s = %.6f" % (flavors[a], flavors[b], value))

P_ee = 0.466982
P_emu = 0.479121
P_etau = 0.053898
P_mue = 0.479121
P_mumu = 0.416326
P_mutau = 0.104553
P_taue = 0.053898
P_taumu = 0.104553
P_tautau = 0.841549


## Pass arrays, do not loop

The single most useful thing to know about performance here: the routines broadcast. A stack of Hamiltonians, an array of baselines, or both, are evaluated in one pass — tens of times faster than the equivalent Python loop, and the results agree to round-off. Notebook 09 measures both the speedup and the agreement.

In [10]:
baselines = np.linspace(1.0, 3000.0, 5)*KM
prob_scan = oscprob3nu.probabilities_3nu(H, baselines)

print("shape:", np.shape(prob_scan))
print("P_ee along the scan: ", np.round(prob_scan[:, 0], 4))
print("all nine at point 0:", np.round(prob_scan[0], 4))

shape: (5, 9)
P_ee along the scan:  [1.     0.9542 0.8969 0.9149 0.9351]
all nine at point 0: [1. 0. 0. 0. 1. 0. 0. 0. 1.]


---

**Next:** [Oscillations in vacuum](02_vacuum_oscillations.ipynb) --- the probabilities against baseline and against energy  
[API reference](https://mbustama.github.io/NuOscProbExact/functions.html) &middot; [Numerical recipes](https://mbustama.github.io/NuOscProbExact/recipes.html) &middot; [All notebooks](.)